**Molecular glue and PROTACs LLM prediction vs. ground truth for the 3rd run**

# Processing Functions

In [ ]:
import requests
import time
from urllib.parse import quote
import pubchempy as pcp


OPSIN_BASE = "https://www.ebi.ac.uk/opsin/ws"
_DASH_CHARS = "‐‑‒–—―−﹣－"
_DASH_TRANS = str.maketrans({c: "-" for c in _DASH_CHARS})


def _normalize_dashes(s):
    if not isinstance(s, str):
        return s
    return s.translate(_DASH_TRANS)


PUBCHEM_BLOCKLIST = {
    "Mei", "Po", "Le",
    "LC-4", "LC-6",
    "D10", "D12", "D15",
    "A13", "13b",
    "SC5", "SC7", "SC10",
    "DBt-10",
    "PS-6",
    "MD9",
    "Cpd 1",
}


def remove_empty_dc50_dmax(df):
    before = len(df)
    cleaned = df.dropna(subset=["DC50", "Dmax"], how="all")
    cleaned = cleaned[~(
        cleaned["DC50"].astype(str).str.strip().isin(["", "nan"])
        & cleaned["Dmax"].astype(str).str.strip().isin(["", "nan"])
    )]
    cleaned = cleaned.reset_index(drop=True)
    after = len(cleaned)
    print(f"Removed {before - after} rows where both DC50 and Dmax are empty")
    print(f"Shape: {before} -> {after}")
    return cleaned


def add_inchikey_opsin(df, sleep_sec=0.1):
    df = df.copy()
    df["Standard_InChIKey"] = pd.NA
    df["Standard_InChIKey_Source"] = pd.NA
    if "SMILES_Source" not in df.columns:
        df["SMILES_Source"] = pd.NA

    has_smiles = df["SMILES"].notna() & (df["SMILES"].astype(str).str.strip() != "")
    df.loc[has_smiles, "SMILES_Source"] = "PAPER"
    print(f"Rows with SMILES from paper: {has_smiles.sum()}")

    success, fail, skip = 0, 0, 0
    failed_rows = []

    for idx, row in df.iterrows():
        iupac = row["IUPAC_Name"]
        if pd.isna(iupac) or str(iupac).strip() == "":
            skip += 1
            continue

        url = f"{OPSIN_BASE}/{quote(str(iupac).strip(), safe='')}.json"
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                inchikey = data.get("stdinchikey")
                smiles = data.get("smiles")
                if inchikey:
                    df.at[idx, "Standard_InChIKey"] = inchikey
                    df.at[idx, "Standard_InChIKey_Source"] = "OPSIN"
                if not has_smiles[idx] and smiles:
                    df.at[idx, "SMILES"] = smiles
                    df.at[idx, "SMILES_Source"] = "OPSIN"
                success += 1
            else:
                failed_rows.append({"row": idx, "status": resp.status_code, "DOI": row["DOI"], "Compound_Name": row["Compound_Name"], "IUPAC_Name": str(iupac)[:100]})
                fail += 1
        except Exception as e:
            failed_rows.append({"row": idx, "error": str(e), "DOI": row["DOI"], "Compound_Name": row["Compound_Name"], "IUPAC_Name": str(iupac)[:100]})
            fail += 1

        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df)} rows...")
        time.sleep(sleep_sec)

    cols = list(df.columns)
    smiles_idx = cols.index("SMILES")
    cols.remove("SMILES_Source")
    cols.insert(smiles_idx + 1, "SMILES_Source")
    df = df[cols]

    print(f"\nDone! Success: {success}, Failed: {fail}, Skipped (no IUPAC): {skip}")
    print(f"Standard_InChIKey filled: {df['Standard_InChIKey'].notna().sum()}")
    print(f"SMILES from PAPER: {(df['SMILES_Source'] == 'PAPER').sum()}")
    print(f"SMILES from OPSIN: {(df['SMILES_Source'] == 'OPSIN').sum()}")
    print(f"SMILES still missing: {df['SMILES'].isna().sum()}")
    if failed_rows:
        print(f"\n--- Failed rows ---")
        for fr in failed_rows:
            print(fr)
    return df


def pubchem_search(df, sleep_sec=0.3, blocklist=None, min_name_length=3):
    """Fill Standard_InChIKey (and IUPAC_Name / SMILES if missing) via PubChem.
    """
    df = df.copy()
    if blocklist is None:
        blocklist = PUBCHEM_BLOCKLIST

    def has_value(v):
        return pd.notna(v) and str(v).strip() != "" and str(v).strip().lower() != "nan"

    def is_numeric_name(s):
        return s.replace(".", "").replace("-", "").isdigit()

    def cname_usable(c):
        if not c:
            return False
        if c in blocklist or _normalize_dashes(c) in blocklist:
            return False
        if is_numeric_name(c):
            return False
        if len(c) < min_name_length:
            return False
        return True

    def search_name_with_synonym(raw_query):
        query = _normalize_dashes(raw_query).strip()
        if not query:
            return None, "empty after normalization"
        try:
            compounds = pcp.get_compounds(query, "name")
            if not compounds:
                return None, "0 results"
            top = compounds[0]
            syn_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{top.cid}/synonyms/JSON"
            syn_resp = requests.get(syn_url, timeout=30)
            if syn_resp.status_code != 200:
                return None, f"synonyms HTTP {syn_resp.status_code} (CID={top.cid})"
            syns = syn_resp.json()["InformationList"]["Information"][0]["Synonym"]
            q_cmp = query.lower()
            if not any(_normalize_dashes(s).lower() == q_cmp for s in syns):
                return None, f"not in synonyms (CID={top.cid})"
            return {"cid": top.cid, "iupac_name": top.iupac_name,
                    "smiles": top.smiles, "inchikey": top.inchikey}, "ok"
        except Exception as e:
            return None, f"error: {e}"

    def search_smiles(query):
        try:
            compounds = pcp.get_compounds(query, "smiles")
            if not compounds:
                return None, "0 results"
            top = compounds[0]
            return {"cid": top.cid, "iupac_name": top.iupac_name,
                    "smiles": top.smiles, "inchikey": top.inchikey}, "ok"
        except Exception as e:
            return None, f"error: {e}"

    def chem_search(iupac, smiles):
        if iupac:
            r, reason = search_name_with_synonym(iupac)
            time.sleep(sleep_sec)
            if r:
                return r, "IUPAC", reason
            iupac_reason = reason
        else:
            iupac_reason = "no IUPAC"
        if smiles:
            r, reason = search_smiles(smiles)
            time.sleep(sleep_sec)
            if r:
                return r, "SMILES", reason
            smiles_reason = reason
        else:
            smiles_reason = "no SMILES"
        return None, None, f"IUPAC={iupac_reason}, SMILES={smiles_reason}"

    for col in ("IUPAC_Name_Source", "Standard_InChIKey",
                "Standard_InChIKey_Source", "SMILES_Source"):
        if col not in df.columns:
            df[col] = pd.NA

    needs_key = df["Standard_InChIKey"].isna() | (df["Standard_InChIKey"].astype(str).str.strip() == "")
    to_search = df[needs_key].index

    queries = {}
    for idx in to_search:
        cname = str(df.at[idx, "Compound_Name"]).strip() if has_value(df.at[idx, "Compound_Name"]) else ""
        iupac = str(df.at[idx, "IUPAC_Name"]).strip()    if has_value(df.at[idx, "IUPAC_Name"])    else ""
        smi   = str(df.at[idx, "SMILES"]).strip()        if has_value(df.at[idx, "SMILES"])        else ""
        queries.setdefault((cname, iupac, smi), []).append(idx)

    print(f"Rows missing Standard_InChIKey: {len(to_search)} ({len(queries)} unique (cname, iupac, smiles) triples)")

    def short(s, n=40):
        return (s[:n] + "...") if s and len(s) > n else s

    cache = {}
    for (cname, iupac, smi), idxs in queries.items():
        usable_c = cname_usable(cname)
        has_chem = bool(iupac) or bool(smi)
        cs = short(cname) if cname else "(no cname)"

        result, method, reason = None, None, ""

        if usable_c and not has_chem:
            r, reason_c = search_name_with_synonym(cname)
            time.sleep(sleep_sec)
            if r:
                result, method, reason = r, "cname", reason_c
            else:
                reason = f"cname={reason_c}"
        elif usable_c and has_chem:
            r, reason_c = search_name_with_synonym(cname)
            time.sleep(sleep_sec)
            if r:
                result, method, reason = r, "cname", reason_c
            else:
                result, method, reason_chem = chem_search(iupac, smi)
                reason = f"cname={reason_c}; {reason_chem}" if not result else f"after cname miss: {reason_chem}"
        elif not usable_c and has_chem:
            result, method, reason_chem = chem_search(iupac, smi)
            reason = f"cname unusable; {reason_chem}"
        else:
            reason = "skip: no usable name and no IUPAC/SMILES"

        cache[(cname, iupac, smi)] = result

        if result:
            print(f"  MATCH '{cs}' via {method}: CID={result['cid']}, InChIKey={result['inchikey']}")
        else:
            print(f"  MISS  '{cs}': {reason}")

    filled_key = filled_iupac = filled_smi = 0
    for (cname, iupac, smi), idxs in queries.items():
        result = cache[(cname, iupac, smi)]
        if result is None:
            continue
        for idx in idxs:
            if result["inchikey"]:
                df.at[idx, "Standard_InChIKey"] = result["inchikey"]
                df.at[idx, "Standard_InChIKey_Source"] = "PUBCHEM"
                filled_key += 1
            if not has_value(df.at[idx, "IUPAC_Name"]) and result.get("iupac_name"):
                df.at[idx, "IUPAC_Name"] = result["iupac_name"]
                df.at[idx, "IUPAC_Name_Source"] = "PUBCHEM"
                filled_iupac += 1
            if not has_value(df.at[idx, "SMILES"]) and result.get("smiles"):
                df.at[idx, "SMILES"] = result["smiles"]
                df.at[idx, "SMILES_Source"] = "PUBCHEM"
                filled_smi += 1

    matched = sum(1 for v in cache.values() if v is not None)
    print(f"\n--- PubChem Results ---")
    print(f"Matched triples: {matched}/{len(queries)}")
    print(f"Standard_InChIKey filled: {filled_key}")
    print(f"IUPAC_Name filled: {filled_iupac}")
    print(f"SMILES filled: {filled_smi}")
    print(f"Standard_InChIKey still missing: {(df['Standard_InChIKey'].isna() | (df['Standard_InChIKey'].astype(str).str.strip() == '')).sum()}")
    return df

def add_connectivity_key(df):
    if 'Standard_InChIKey' in df.columns:
        df['Connectivity_Key'] = df['Standard_InChIKey'].str[:14]
    return df

# Porcess PROTACs Extracted Data

In [2]:
import pandas as pd
import os

protac_path = '/Users/yaochenr/project/molecular_glue_extractor/output/results/260502_0508/data_processed.csv'
merged_protacs = pd.read_csv(protac_path)
merged_protacs.head()

,DOI,source,Compound_Name,IUPAC_Name,SMILES,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc
0,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",NaN,KRASG12C,VHL,immunoblotting,NCI-H2030,590.0 ± 200.0,nM,24,~80,24,2.5 μM
1,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",NaN,KRASG12C,VHL,immunoblotting,MIA PaCa-2,320.0 ± 80.0,nM,24,~75,24,NaN
2,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",NaN,KRASG12C,VHL,immunoblotting,SW1573,760.0 ± 300.0,nM,24,~90,24,NaN
3,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",NaN,KRASG12C,VHL,immunoblotting,NCI-H23,250.0 ± 80.0,nM,24,~90,24,NaN
4,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",NaN,KRASG12C,VHL,immunoblotting,NCI-H358,520.0 ± 300.0,nM,24,~40,24,NaN


In [4]:
data_dir = "/Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd"
os.makedirs(data_dir, exist_ok=True)
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_3rd.csv")
merged_protacs.to_csv(output_path, index=False)
print(f"Saved merged data to: {output_path}")
print(f"Shape: {merged_protacs.shape}")

Saved merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd/260502_merged_protac_data_processed_3rd.csv
Shape: (809, 15)


### Remove rows where DC50 & Dmax are both empty

In [6]:
merged_cleaned_protac = remove_empty_dc50_dmax(merged_protacs)

Removed 115 rows where both DC50 and Dmax are empty
Shape: 809 -> 694


In [ ]:
output_merged = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_3rd.csv")
merged_cleaned_protac.to_csv(output_merged, index=False)
print(f"Saved cleaned merged data to: {output_merged}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved cleaned merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd/260502_merged_protac_data_processed_cleaned_3rd.csv
Shape: (694, 15)


### Add Standard InChIKey via OPSIN

In [8]:
merged_cleaned_protac = add_inchikey_opsin(merged_cleaned_protac)

Rows with SMILES from paper: 126
Processed 150/694 rows...
Processed 300/694 rows...
Processed 450/694 rows...

Done! Success: 387, Failed: 101, Skipped (no IUPAC): 206
Standard_InChIKey filled: 387
SMILES from PAPER: 126
SMILES from OPSIN: 282
SMILES still missing: 286

--- Failed rows ---
{'row': 20, 'status': 404, 'DOI': '10.1038/s41467-018-08027-7', 'Compound_Name': 'SJFα', 'IUPAC_Name': 'N-(3-Fluoro-4-((7-(4-(4-(2-(((S)-1-((2S,4R)-4-hydroxy-2-((4-(4-methylthiazol-5-yl)benzyl) carbamoyl)'}
{'row': 21, 'status': 404, 'DOI': '10.1038/s41467-018-08027-7', 'Compound_Name': 'SJFα', 'IUPAC_Name': 'N-(3-Fluoro-4-((7-(4-(4-(2-(((S)-1-((2S,4R)-4-hydroxy-2-((4-(4-methylthiazol-5-yl)benzyl) carbamoyl)'}
{'row': 25, 'status': 404, 'DOI': '10.1038/s41467-018-08027-7', 'Compound_Name': 'SJF-6690', 'IUPAC_Name': 'N-(3-Fluoro-4-((7-(2-(3-(((S)-1-((2S,4R)-4-hydroxy-2-((4-(4-methylthiazol-5-yl)benzyl) carbamoyl)pyr'}
{'row': 26, 'status': 404, 'DOI': '10.1038/s41467-018-08027-7', 'Compound_Name': 'S

In [9]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_3rd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd/260502_merged_protac_data_processed_cleaned_inchikey_3rd.csv
Shape: (694, 18)


### PubChem search

In [10]:
merged_cleaned_protac = pubchem_search(merged_cleaned_protac)

Rows missing Standard_InChIKey: 307 (139 unique (cname, iupac, smiles) triples)
  MATCH 'DT2216' via cname: CID=139331475, InChIKey=PXVFFBGSTYQHRO-REQIQPEASA-N
  MISS  'D3': skip: no usable name and no IUPAC/SMILES
  MISS  'D10': skip: no usable name and no IUPAC/SMILES
  MISS  'D12': skip: no usable name and no IUPAC/SMILES
  MISS  'D14': cname=0 results
  MISS  'D15': skip: no usable name and no IUPAC/SMILES
  MISS  'SJFα': cname=0 results; IUPAC=0 results, SMILES=no SMILES
  MATCH 'SJF-6690' via cname: CID=145748096, InChIKey=PCRYOGWRHSLIEZ-XZVMHGATSA-N
  MATCH 'SJF-6677' via cname: CID=145748084, InChIKey=LCBLIBYLVHDHMI-JCYIICMFSA-N
  MATCH 'SJF-0628' via cname: CID=146547699, InChIKey=MBCAVOCJJAQHHT-FGUOTCRGSA-N
  MISS  '15a': cname=0 results; IUPAC=0 results, SMILES=no SMILES
  MISS  '15b': cname=0 results; IUPAC=0 results, SMILES=no SMILES
  MATCH 'RC32' via cname: CID=156599864, InChIKey=XCSUEITWRRFFMN-LPIXFUNRSA-N
  MISS  'CRBN_1b': cname=0 results
  MISS  'CRBN_1e': cname=0 r

In [11]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_pubchem_3rd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd/260502_merged_protac_data_processed_cleaned_inchikey_pubchem_3rd.csv
Shape: (694, 19)


### Add connectivity_key

In [12]:
merged_cleaned_protac = add_connectivity_key(merged_cleaned_protac)

In [13]:
merged_cleaned_protac

,DOI,source,Compound_Name,IUPAC_Name,SMILES,SMILES_Source,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,Standard_InChIKey,Standard_InChIKey_Source,IUPAC_Name_Source,Connectivity_Key
0,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",ClC=1C=CC=C2C=CC=C(C12)N1CC=2N=C(N=C(C2CC1)N1C...,OPSIN,KRASG12C,VHL,immunoblotting,NCI-H2030,590.0 ± 200.0,nM,24,~80,24,2.5 μM,ZCGQZLKPUVGCBQ-HLMPTVQRSA-N,OPSIN,<NA>,ZCGQZLKPUVGCBQ
1,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",ClC=1C=CC=C2C=CC=C(C12)N1CC=2N=C(N=C(C2CC1)N1C...,OPSIN,KRASG12C,VHL,immunoblotting,MIA PaCa-2,320.0 ± 80.0,nM,24,~75,24,NaN,ZCGQZLKPUVGCBQ-HLMPTVQRSA-N,OPSIN,<NA>,ZCGQZLKPUVGCBQ
2,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",ClC=1C=CC=C2C=CC=C(C12)N1CC=2N=C(N=C(C2CC1)N1C...,OPSIN,KRASG12C,VHL,immunoblotting,SW1573,760.0 ± 300.0,nM,24,~90,24,NaN,ZCGQZLKPUVGCBQ-HLMPTVQRSA-N,OPSIN,<NA>,ZCGQZLKPUVGCBQ
3,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",ClC=1C=CC=C2C=CC=C(C12)N1CC=2N=C(N=C(C2CC1)N1C...,OPSIN,KRASG12C,VHL,immunoblotting,NCI-H23,250.0 ± 80.0,nM,24,~90,24,NaN,ZCGQZLKPUVGCBQ-HLMPTVQRSA-N,OPSIN,<NA>,ZCGQZLKPUVGCBQ
4,10.1021/acscentsci.0c00411,supplementary_oc0c00411_si_001.md,LC-2,"(2S,4R)-1-((S)-2-(3-(3-((S)-2-(((7-(8-chlorona...",ClC=1C=CC=C2C=CC=C(C12)N1CC=2N=C(N=C(C2CC1)N1C...,OPSIN,KRASG12C,VHL,immunoblotting,NCI-H358,520.0 ± 300.0,nM,24,~40,24,NaN,ZCGQZLKPUVGCBQ-HLMPTVQRSA-N,OPSIN,<NA>,ZCGQZLKPUVGCBQ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
689,10.1039/d3sc04629j,main_text,WJ112-14,NaN,NaN,<NA>,PI3Kβ (p110β),CRBN,PRM targeted proteomics,MCF7,NaN,NaN,NaN,0,6,1 μM,<NA>,<NA>,<NA>,<NA>
690,10.1039/d3sc04629j,main_text,WJ204-12,NaN,NaN,<NA>,PI3Kβ (p110β),CRBN,PRM targeted proteomics,MCF7,NaN,NaN,NaN,~50,6,100 nM,<NA>,<NA>,<NA>,<NA>
691,10.1038/s41467-019-11429-w,supplementary_41467_2019_11429_MOESM1_ESM.md,DGY-03-081,"N2-((S)-1-cyclohexyl-2-((S)-1-((1S,3aR,6aS)-1-...",C1(CCCCC1)[C@@H](C(=O)N[C@H](C(=O)N1[C@@H]([C@...,OPSIN,HCV NS3,CRBN,Flow cytometry of NS3-eGFP-2A-mCherry reporter...,Flp-In T-REx 293,668.5,nM,4 h,NaN,NaN,NaN,KJEYLCKKOBCAHO-KSESKIPOSA-N,OPSIN,<NA>,KJEYLCKKOBCAHO
692,10.1038/s41467-019-11429-w,supplementary_41467_2019_11429_MOESM1_ESM.md,DGY-04-035,"N2-((S)-1-cyclohexyl-2-((S)-1-((1S,3aR,6aS)-1-...",C1(CCCCC1)[C@@H](C(=O)N[C@H](C(=O)N1[C@@H]([C@...,OPSIN,HCV NS3,CRBN,Flow cytometry of NS3-eGFP-2A-mCherry reporter...,Flp-In T-REx 293,488.6,nM,4 h,NaN,NaN,NaN,NPXHZAXKZQRUGW-KSESKIPOSA-N,OPSIN,<NA>,NPXHZAXKZQRUGW


In [14]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_pubchem_ck_3rd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_3rd/260502_merged_protac_data_processed_cleaned_inchikey_pubchem_ck_3rd.csv
Shape: (694, 20)


# Porcess Molecular Glues Extracted Data

In [2]:
import pandas as pd
import os

glue_path = '/Users/yaochenr/project/molecular_glue_extractor/output/results/260504_1725/data_processed.csv'
merged_glues = pd.read_csv(glue_path)
merged_glues.head()

,DOI,source,Compound_Name,IUPAC_Name,SMILES,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc
0,10.1038/ncomms15398,main_text,lenalidomide,NaN,NaN,ZFP91,CRBN (CRL4CRBN),pSILAC mass spectrometry (multi time point; an...,HEK293T,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1038/ncomms15398,main_text,lenalidomide,NaN,NaN,ZFP91,CRBN (CRL4CRBN),pSILAC mass spectrometry (single time point),Hct116,NaN,NaN,NaN,NaN,NaN,NaN
2,10.1038/ncomms15398,main_text,lenalidomide,NaN,NaN,CSNK1A1,CRBN (CRL4CRBN),pSILAC mass spectrometry (multi time point; an...,HEK293T,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1038/ncomms15398,main_text,lenalidomide,NaN,NaN,CSNK1A1,CRBN (CRL4CRBN),pSILAC mass spectrometry (single time point),Hct116,NaN,NaN,NaN,NaN,NaN,NaN
4,10.1038/ncomms15398,main_text,lenalidomide,NaN,NaN,ZFP91,CRBN (CRL4CRBN),Western blot,MM.1S,NaN,NaN,NaN,NaN,NaN,NaN


### Remove rows where DC50 & Dmax are both empty

In [3]:
merged_cleaned_glues = remove_empty_dc50_dmax(merged_glues)

Removed 122 rows where both DC50 and Dmax are empty
Shape: 401 -> 279


In [4]:
data_dir_glues = "/Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_3rd"
os.makedirs(data_dir_glues, exist_ok=True)
output_merged = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_3rd.csv")
merged_cleaned_glues.to_csv(output_merged, index=False)
print(f"Saved cleaned merged data to: {output_merged}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved cleaned merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_3rd/260504_merged_glue_data_processed_cleaned_3rd.csv
Shape: (279, 15)


## Add Standard InChIKey via OPSIN

In [5]:
merged_cleaned_glues = add_inchikey_opsin(merged_cleaned_glues)

Rows with SMILES from paper: 32
Processed 250/279 rows...

Done! Success: 208, Failed: 13, Skipped (no IUPAC): 58
Standard_InChIKey filled: 208
SMILES from PAPER: 32
SMILES from OPSIN: 178
SMILES still missing: 69

--- Failed rows ---
{'row': 59, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '7', 'IUPAC_Name': "3-(6'-oxo-4-phenoxy-6',8'-dihydro-2'H,7'H-spiro[cyclohexane-1,3' furo[2,3-e]isoindol]-7'-yl)piperidi"}
{'row': 61, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '9', 'IUPAC_Name': "3-(1-benzyl-6'-oxo-6',8'-dihydro-2'H,7'H-spiro[azepane-4,3' furo[2,3-e]isoindol]-7'-yl)piperidine-2,"}
{'row': 64, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '12', 'IUPAC_Name': "3-(1'-benzyl-3',3'-difluoro-6-oxo-6,8-dihydro-2H,7H-spiro[furo[2,3 e]isoindole-3,4'-piperidin]-7-yl)"}
{'row': 65, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '13', 'IUPAC_Name': "3-(1'-(cyclohexylmethyl)-6-oxo-6,8-dihydro-2H,7H-s

In [6]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_3rd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_3rd/260504_merged_glue_data_processed_cleaned_inchikey_3rd.csv
Shape: (279, 18)


## PubChem Search

In [7]:
merged_cleaned_glues = pubchem_search(merged_cleaned_glues)

Rows missing Standard_InChIKey: 71 (45 unique (cname, iupac, smiles) triples)
  MISS  'MG-Lin1': cname=0 results
  MATCH 'lenalidomide' via cname: CID=216326, InChIKey=GOTYRUGSSMKFNF-UHFFFAOYSA-N
  MATCH 'dBET57' via cname: CID=118912822, InChIKey=CZRLOIDJCMKJHE-UXMRNZNESA-N
  MATCH 'A6' via SMILES: CID=166642468, InChIKey=FQLCLMHMOFLVSV-UHFFFAOYSA-N
  MISS  '13-7': skip: no usable name and no IUPAC/SMILES
  MATCH 'DKY709' via cname: CID=137519326, InChIKey=OMISHRJQMYQPMG-UHFFFAOYSA-N
  MISS  '7': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '9': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '12': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '13': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '14': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '15': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  'iDeg-6': cname=0 results
  MATCH 'Pomalidomide' via cname: CID=134780, InChIKey=UVSMNLNDYGZFPF-UHFFFAOYSA-N
  MATCH 'Le

In [8]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_pubchem_3rd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_3rd/260504_merged_glue_data_processed_cleaned_inchikey_pubchem_3rd.csv
Shape: (279, 19)


## Add connectivity_key

In [9]:
merged_cleaned_glues = add_connectivity_key(merged_cleaned_glues)

In [10]:
merged_cleaned_glues[['Compound_Name', 'Standard_InChIKey', 'Connectivity_Key']]

,Compound_Name,Standard_InChIKey,Connectivity_Key
0,MG-Lin1,<NA>,<NA>
1,5a,MSHTVRYXGFREAN-UHFFFAOYSA-N,MSHTVRYXGFREAN
2,7d,QRQMHYISDDHZBY-UHFFFAOYSA-N,QRQMHYISDDHZBY
3,7f,NSEUGMATEWSNTO-UHFFFAOYSA-N,NSEUGMATEWSNTO
4,lenalidomide,GOTYRUGSSMKFNF-UHFFFAOYSA-N,GOTYRUGSSMKFNF
...,...,...,...
274,CCT373566,GSGDUDAFETZSPM-IATAILRESA-N,GSGDUDAFETZSPM
275,CCT373567,GSGDUDAFETZSPM-DHZVRSILSA-N,GSGDUDAFETZSPM
276,CCT373566,GSGDUDAFETZSPM-IATAILRESA-N,GSGDUDAFETZSPM
277,CCT373567,GSGDUDAFETZSPM-DHZVRSILSA-N,GSGDUDAFETZSPM


In [11]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_pubchem_ck_3rd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_3rd/260504_merged_glue_data_processed_cleaned_inchikey_pubchem_ck_3rd.csv
Shape: (279, 20)
